# Ward-42 Survey OCR Pipeline — Kaggle Run

**SETUP:** Settings → Accelerator → GPU T4 x2  
**INPUTS:** `rk9098/ward42-surveys` + `rk9098/ocr-platform-code`  
**OUTPUTS:** `ocr_results.csv`, `evidence.db`, `run_manifest.json`

Run all cells in order. Expected: 2–4 hours on T4 for 1446 PDFs.

In [ ]:
%%capture --no-display

# STEP 1: Install dependencies
!pip install -q --upgrade pip

!pip install -q \
    pypdfium2 rapidfuzz Pillow pandas numpy \
    opencv-contrib-python pydantic pyyaml

# PyTorch CUDA 12.1 (T4)
!pip install -q \
    torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu121

# PaddlePaddle GPU — pinned to avoid pipeline_1f1b import bug
!pip install -q "paddlepaddle-gpu==3.0.0.post119"
!pip install -q "paddleocr>=3.0,<3.1"

# MS-SWIFT + VLM deps
!pip install -q ms-swift transformers accelerate sentencepiece

# Install ocr-platform from code dataset
!pip install -q -e /kaggle/input/ocr-platform-code --no-deps 2>/dev/null || \
  (cp -r /kaggle/input/ocr-platform-code/src ./src && \
   cp -r /kaggle/input/ocr-platform-code/configs ./configs && \
   cp -r /kaggle/input/ocr-platform-code/templates ./templates && \
   pip install -q -e . --no-deps)

print('Dependencies installed.')
!nvidia-smi -L

In [ ]:
# PaddlePaddle pipeline_1f1b compatibility patch
import os, sys

pass_utils = os.path.join(
    sys.prefix, 'lib',
    'python' + '.'.join(map(str, sys.version_info[:2])),
    'site-packages', 'paddle', 'distributed', 'passes', 'pass_utils.py'
)
if os.path.exists(pass_utils):
    src_text = open(pass_utils).read()
    if 'def forward_complete_op_role' not in src_text:
        with open(pass_utils, 'a') as f:
            f.write('\n\ndef forward_complete_op_role(op):\n    return 0\n')
        print(f'Patched pass_utils')
    else:
        print('Patch already applied')

import paddle
print(f'PaddlePaddle {paddle.__version__} | GPU: {paddle.device.is_compiled_with_cuda()}')
from paddleocr import PaddleOCR
PaddleOCR(lang='en')
print('PaddleOCR (en) ready')
try:
    PaddleOCR(lang='hi')
    print('PaddleOCR (hi) ready')
except Exception:
    PaddleOCR(lang='multilingual')
    print('PaddleOCR (multilingual) ready')

In [ ]:
# HindiOCR-VLM adapter check
import os, subprocess
from pathlib import Path

ADAPTER_DIR = '/kaggle/working/hindi_vlm_adapter'
HINDI_VLM = False

result = subprocess.run(
    ['huggingface-cli', 'download', '--repo-type', 'model',
     '--local-dir', ADAPTER_DIR,
     '--local-dir-use-symlinks', 'False', '--quiet',
     'shaoncvit/HindiOCR-VLM'],
    capture_output=True, text=True, timeout=300
)
if result.returncode == 0 and (Path(ADAPTER_DIR) / 'adapter_config.json').exists():
    print(f'HindiOCR-VLM adapter available: {ADAPTER_DIR}')
    HINDI_VLM = True
    os.environ['HINDI_VLM_ADAPTER_DIR'] = ADAPTER_DIR
else:
    print('HindiOCR-VLM not available — using PaddleOCR only')
    os.environ['USE_HINDI_VLM'] = '0'

In [ ]:
# STEP 3: Run the full pipeline on all PDFs
import os, sys, json, time, shutil
from pathlib import Path
from datetime import datetime

sys.path.insert(0, '/kaggle/input/ocr-platform-code/src')

PDF_ROOT  = Path('/kaggle/input/ward42-surveys')
CODE_ROOT = Path('/kaggle/input/ocr-platform-code')
OUT_DIR   = Path('/kaggle/working/output')
DB_PATH   = OUT_DIR / 'evidence.db'
TEMPLATE  = CODE_ROOT / 'configs' / 'ward42_template.json'

for d in [OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

pdfs = sorted(PDF_ROOT.rglob('*.pdf'))
print(f'Found {len(pdfs)} PDFs in {PDF_ROOT}')

from ocr_platform.orchestrator.pipeline import SurveyPipeline

pipeline = SurveyPipeline(
    template_path=str(TEMPLATE),
    db_path=str(DB_PATH),
    output_dir=str(OUT_DIR),
    dpi=150,
    use_hindi_vlm=HINDI_VLM,
)

results  = []
errors   = []
t0 = time.time()

for i, pdf_path in enumerate(pdfs):
    doc_id = pdf_path.stem
    rel    = pdf_path.relative_to(PDF_ROOT)
    print(f'\n{"="*60}')
    print(f'[{i+1}/{len(pdfs)}] {rel}')
    print(f'{"="*60}')

    try:
        res = pipeline.run(pdf_path)
        results.append(res)

        stats = res.get('stats', {})
        print(f"  cells={stats.get('cells',0)} "
              f"auto={stats.get('auto',0)} "
              f"flagged={stats.get('low_conf',0)} "
              f"qa={res.get('qa_anomalies',0)}")

    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback; traceback.print_exc()
        errors.append({'doc_id': doc_id, 'error': str(e)})

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f'COMPLETE: {len(pdfs)-len(errors)}/{len(pdfs)} PDFs in {elapsed:.0f}s ({elapsed/60:.1f}min)')
print(f'Errors: {len(errors)}')

# --- Read output CSV from export dir (written as side-effect of pipeline.run) ---
csv_dir = OUT_DIR / 'csv'
long_csv_path = csv_dir / 'extracted_cells_long.csv'
master_csv = OUT_DIR / 'ocr_results.csv'

if long_csv_path.exists():
    shutil.copy2(long_csv_path, master_csv)
    print(f'Master CSV: {master_csv} ({master_csv.stat().st_size // 1024} KB)')
else:
    print('WARNING: no long CSV found — check pipeline output')

# --- Write run manifest ---
manifest = {
    'run_id': f'kaggle_{datetime.utcnow():%Y%m%d_%H%M%S}',
    'timestamp': datetime.utcnow().isoformat() + 'Z',
    'total_pdfs': len(pdfs),
    'successful': len(pdfs) - len(errors),
    'errors': len(errors),
    'elapsed_seconds': round(elapsed, 1),
    'hindi_vlm': HINDI_VLM,
    'error_list': errors,
}
with open(OUT_DIR / 'run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2, default=str)

print(f'DB: {DB_PATH}')
print(f'Manifest: {OUT_DIR / "run_manifest.json"}')
print('Done! Download /kaggle/working/output/ from the Output tab.')